# 3.4 Orchestrating Multi-Agent Workflows

**Week 4 — Agentic AI & Multi-Agent Systems**

## Learning objectives
- Implement a **fixed-sequence** multi-agent flow (AutoGen's `RoundRobinGroupChat` pattern)
- Implement a **dynamic, LLM-routed** flow (AutoGen's `SelectorGroupChat` pattern)
- Understand CrewAI-style orchestration: Chain-of-Thought reasoning, execution layers, Flow orchestrators
- Save an agent's conversational state to JSON and reload it into another agent/session


## 1. Fixed Sequence — `RoundRobinGroupChat` Style

Some workflows have a **known, unchanging order**. A classic example: a Teacher agent poses a question,
a Student agent answers, the Teacher gives feedback, repeat. AutoGen's `RoundRobinGroupChat` implements
exactly this — each agent gets a turn in a fixed rotation, regardless of what was said.

```python
# Real AutoGen usage:
from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat([teacher_agent, student_agent], max_turns=6)
result = await team.run(task="Practice: capital cities of Europe")
```


In [ ]:
from dataclasses import dataclass
from typing import List, Callable

@dataclass
class SimpleAgent:
    name: str
    respond: Callable[[str], str]

def teacher_respond(last_message: str) -> str:
    if "capital" not in last_message.lower():
        return "Teacher: What is the capital of Germany?"
    return f"Teacher: Correct feedback on -> '{last_message}'"

def student_respond(last_message: str) -> str:
    if "germany" in last_message.lower():
        return "Student: The capital of Germany is Berlin."
    return "Student: I'm ready for the next question."

def round_robin_chat(agents: List[SimpleAgent], max_turns: int, opening: str):
    transcript = [opening]
    print(opening)
    for i in range(max_turns):
        agent = agents[i % len(agents)]
        reply = agent.respond(transcript[-1])
        transcript.append(reply)
        print(reply)
    return transcript

teacher = SimpleAgent("Teacher", teacher_respond)
student = SimpleAgent("Student", student_respond)

_ = round_robin_chat([teacher, student], max_turns=4, opening="Teacher: Let's begin the quiz.")


Notice the rotation is **fixed**: Teacher, Student, Teacher, Student — no matter what either agent
says. This predictability is exactly what you want for structured, sequential tasks like quizzes,
scripted interviews, or a fixed pipeline stage order.


## 2. Dynamic Decision-Making — `SelectorGroupChat` Style

Other workflows need the **next speaker to depend on content**, not a fixed rotation. AutoGen's
`SelectorGroupChat` uses an LLM to read the conversation so far and pick which agent should go next —
e.g. a Researcher → Writer → Critic pipeline where the Critic might send work back to the Writer instead
of always advancing forward.

```python
# Real AutoGen usage:
from autogen_agentchat.teams import SelectorGroupChat
team = SelectorGroupChat([researcher_agent, writer_agent, critic_agent], model_client=model_client)
result = await team.run(task="Write a short brief on RAG evaluation metrics")
```


In [ ]:
def mock_selector(transcript: List[str], agents: List[str]) -> str:
    """Stands in for the LLM-based router in SelectorGroupChat. In production this is a real model
    call that reads the whole transcript and decides who should speak next."""
    last = transcript[-1].lower()
    if last.startswith("task:"):
        return "Researcher"
    if last.startswith("researcher:"):
        return "Writer"
    if last.startswith("writer:") and "draft" in last:
        return "Critic"
    if last.startswith("critic:") and "revise" in last:
        already_revised = any("revised draft" in t.lower() for t in transcript)
        return "DONE" if already_revised else "Writer"   # dynamic: sent back to Writer once, then done
    if last.startswith("writer:") and "revised draft" in last:
        return "DONE"
    return "DONE"

def agent_step(name: str, transcript: List[str]) -> str:
    if name == "Researcher":
        return "Researcher: Key facts gathered on RAGAS metrics -> faithfulness, relevancy, precision, recall."
    if name == "Writer":
        if any("revise" in t.lower() for t in transcript):
            return "Writer: Revised draft -> a tighter 3-sentence brief on the four RAGAS metrics."
        return "Writer: Draft -> RAGAS scores four things about a RAG pipeline's answers."
    if name == "Critic":
        return "Critic: Draft is too vague, please revise for specificity."
    return ""

def selector_group_chat(agents: List[str], task: str, max_turns: int = 6):
    transcript = [f"Task: {task}"]
    print(transcript[0])
    for _ in range(max_turns):
        nxt = mock_selector(transcript, agents)
        if nxt == "DONE":
            print("-> conversation concluded by selector")
            break
        msg = agent_step(nxt, transcript)
        transcript.append(msg)
        print(msg)
    return transcript

_ = selector_group_chat(["Researcher", "Writer", "Critic"], "Write a short brief on RAG evaluation metrics")


Compare this to the fixed round-robin above: the **Critic's feedback changed the routing** — the
selector sent control back to the Writer instead of ending the conversation. This is the essence of
*dynamic* orchestration: the next step depends on the content of the conversation, decided by an LLM
acting as router.


## 3. CrewAI Orchestration

CrewAI frames the same underlying loop as a **"crew"** of role-based agents managed by an orchestration
layer:

- **Chain-of-Thought reasoning** — each agent reasons step by step before acting, same as ReAct.
- **Execution layers** — the actual tool/code execution is handled separately from the reasoning agent
  (same Brain/Hands split from 3.1–3.2).
- **Flow orchestrators** — a higher-level object sequences which agent runs when, optionally
  conditionally branching based on a previous agent's output (similar to `SelectorGroupChat`, but
  expressed declaratively as a "Flow" rather than an LLM router).

```python
# Real CrewAI usage sketch:
from crewai import Agent, Task, Crew, Flow

researcher = Agent(role="Researcher", goal="Gather accurate facts", backstory="...")
writer = Agent(role="Writer", goal="Produce a clear brief", backstory="...")

crew = Crew(agents=[researcher, writer],
            tasks=[Task(description="Research RAGAS", agent=researcher),
                   Task(description="Write brief from research", agent=writer)])
result = crew.kickoff()
```

The key conceptual difference from AutoGen: CrewAI's orchestration is **declared up front** (you define
the crew and its tasks as configuration), whereas AutoGen's `SelectorGroupChat` decides routing
**dynamically, at runtime**, via an LLM call on every turn.


## 4. State and Memory Management — Saving and Reloading Agent State

Multi-agent systems often need to **hand off context across process boundaries** — e.g. one agent runs
in a batch job overnight, and a different agent (or a different session entirely) needs to pick up
where it left off. The fix is to serialise the running state to JSON and reload it.


In [ ]:
import json
from pathlib import Path

def save_agent_state(state: dict, path: str):
    Path(path).write_text(json.dumps(state, indent=2))
    print(f"Saved state to {path}")

def load_agent_state(path: str) -> dict:
    state = json.loads(Path(path).read_text())
    print(f"Loaded state from {path}")
    return state

# Simulate agent A finishing its part of the work and persisting state
agent_a_state = {
    "conversation_id": "conv-8841",
    "classification": {"request_type": "refund_query", "urgency": "high"},
    "grounded_context": "Refunds are processed within 5-7 business days per the returns policy.",
    "history": ["Researcher: gathered refund policy context"]
}
save_agent_state(agent_a_state, "/tmp/agent_state.json")

# A completely different agent/session picks up the handoff later
agent_b_state = load_agent_state("/tmp/agent_state.json")
print("\nAgent B resumes with:")
for k, v in agent_b_state.items():
    print(f"  {k}: {v}")


This is the same pattern that Week 5's MCP servers and Week 6's Nexus capstone rely on for
durable, auditable hand-offs between components that don't share memory — instead of an in-process
Python dict, production systems typically persist this JSON to a database, a message queue, or an MCP
resource.


## Key Takeaways

- **Fixed-sequence orchestration** (round-robin) suits tasks with a known, unchanging turn order.
- **Dynamic orchestration** (selector-based) suits tasks where the next step depends on what was just
  said — an LLM acts as router.
- **CrewAI** declares the team and task sequence up front (configuration-first); **AutoGen's selector**
  decides routing live, at runtime.
- Persisting agent state to JSON is what makes multi-agent hand-offs durable across process or session
  boundaries.

## Check your understanding
1. When would you prefer a fixed round-robin sequence over a dynamic selector, and why?
2. In the CrewAI sketch, what plays the role of the "Hands" from the Brain + Hands analogy?
3. Why serialise agent state to JSON instead of just keeping it in a Python variable?

Next: **3.5 Practical Multi-Agent Implementations** — concrete CrewAI examples, observability, and
human-in-the-loop design.
